In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
source_file=f"{landing_folder_path}/{v_batch_id}/races.csv"
table_name=f"{catalog_name}.{bronze_schema}.races"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DateType

races_Schema = StructType([
    StructField('season',IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('raceName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType()),
    
]) 




In [0]:
races_df=(
    spark.read
    .format('csv')
    .option('header','true')
    #.option('inferSchema','true')
    .option('mode','FAILFAST')
    .schema(races_Schema)
    .load(source_file)
)

In [0]:
races_df.show(5)

In [0]:
display(races_df)

In [0]:
from pyspark.sql import functions as F
races_final_df =  add_ingestion_metadata(races_df)

In [0]:
display(races_final_df)

In [0]:
# (
#     races_final_df
#     .write
#     .format('delta')
#     .mode('overwrite')
#     .saveAsTable(table_name)
# )
write_to_bronze(
    input_df=races_final_df,
    target_table=table_name,
    batch_id=v_batch_id
)

In [0]:
display(spark.table(table_name))

In [0]:
display(spark.table('formula1.bronze.circuits'))